# BB8 — Phase 1: Tokenization Demo

This notebook explores and compares the three tokenization strategies implemented in BB8:

1. **Character-level** — one token per character
2. **Word-level** — one token per word
3. **Byte Pair Encoding (BPE)** — learned subword units

We'll train each tokenizer on the Tiny Shakespeare dataset and compare:
- Vocabulary size
- Sequence compression
- Token distributions
- Encode/decode quality

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 4)

from tokenizer import CharTokenizer, WordTokenizer, BPETokenizer

print('Imports OK')

## Load Training Data

In [ ]:
DATA_PATH = '../data/tiny_shakespeare.txt'

try:
    with open(DATA_PATH, 'r', encoding='utf-8') as f:
        text = f.read()
    print(f'Loaded {len(text):,} characters')
    print(f'Unique characters: {len(set(text))}')
    print(f'\nFirst 500 characters:')
    print(text[:500])
except FileNotFoundError:
    print('Data not found. Run: python data/prepare_data.py --dataset tiny_shakespeare')
    # Use sample text for demonstration
    text = """
    First Citizen: Before we proceed any further, hear me speak.
    All: Speak, speak.
    First Citizen: You are all resolved rather to die than to famish?
    All: Resolved. resolved.
    First Citizen: First, you know Caius Marcius is chief enemy to the people.
    All: We know't, we know't.
    """ * 100
    print(f'Using sample text ({len(text):,} characters)')

## 1. Character-Level Tokenizer

In [ ]:
char_tok = CharTokenizer()
char_tok.train([text])

sample = 'To be, or not to be, that is the question.'
char_ids = char_tok.encode(sample)

print(f'Vocabulary size: {char_tok.get_vocab_size()}')
print(f'\nSample text: {repr(sample)}')
print(f'Token IDs: {char_ids}')
print(f'Num tokens: {len(char_ids)} (= {len(sample)} characters)')
print(f'\nDecoded: {repr(char_tok.decode(char_ids))}')

In [ ]:
# Show character vocabulary
printable = {k: v for k, v in char_tok.vocab.items() if len(k) == 1}
print('Character vocabulary (first 30):')
for char, idx in sorted(printable.items(), key=lambda x: x[1])[:30]:
    display_char = repr(char) if char in (' ', '\n', '\t') else char
    print(f'  {idx:3d} → {display_char}')

## 2. Word-Level Tokenizer

In [ ]:
word_tok = WordTokenizer()
word_tok.train([text], vocab_size=5000)

word_ids = word_tok.encode(sample)

print(f'Vocabulary size: {word_tok.get_vocab_size()}')
print(f'\nSample text: {repr(sample)}')
print(f'Token IDs: {word_ids}')
print(f'Num tokens: {len(word_ids)}')
print(f'Compression ratio: {len(sample)/len(word_ids):.1f}x')
print(f'\nDecoded: {repr(word_tok.decode(word_ids))}')

## 3. BPE Tokenizer

In [ ]:
bpe_tok = BPETokenizer()
bpe_tok.train([text], vocab_size=3000)

bpe_ids = bpe_tok.encode(sample)

print(f'Vocabulary size: {bpe_tok.get_vocab_size()}')
print(f'Merge rules learned: {len(bpe_tok.merges)}')
print(f'\nSample text: {repr(sample)}')
print(f'Token IDs: {bpe_ids}')
print(f'Num tokens: {len(bpe_ids)}')
print(f'Compression ratio: {len(sample)/len(bpe_ids):.1f}x')
print(f'\nDecoded: {repr(bpe_tok.decode(bpe_ids))}')

## Comparison: Vocabulary Size & Sequence Length

In [ ]:
# Encode the full corpus with each tokenizer and compare stats
full_char = char_tok.encode(text)
full_word = word_tok.encode(text)
full_bpe  = bpe_tok.encode(text)

tokenizers = ['Char', 'Word (5k)', 'BPE (3k)']
vocab_sizes = [char_tok.get_vocab_size(), word_tok.get_vocab_size(), bpe_tok.get_vocab_size()]
seq_lengths = [len(full_char), len(full_word), len(full_bpe)]
compressions = [len(text)/l for l in seq_lengths]

print('Tokenizer Comparison')
print('='*60)
for name, vs, sl, cr in zip(tokenizers, vocab_sizes, seq_lengths, compressions):
    print(f'{name:15s}  vocab={vs:6,}  tokens={sl:9,}  compression={cr:.2f}x')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(tokenizers, vocab_sizes, color=['#4e79a7', '#f28e2b', '#e15759'])
axes[0].set_title('Vocabulary Size')
axes[0].set_ylabel('Number of tokens')

axes[1].bar(tokenizers, seq_lengths, color=['#4e79a7', '#f28e2b', '#e15759'])
axes[1].set_title('Corpus Length (tokens)')
axes[1].set_ylabel('Number of tokens')

axes[2].bar(tokenizers, compressions, color=['#4e79a7', '#f28e2b', '#e15759'])
axes[2].set_title('Compression Ratio')
axes[2].set_ylabel('Characters per token')

plt.tight_layout()
plt.savefig('../outputs/tokenizer_comparison.png', dpi=150)
plt.show()
print('Saved: outputs/tokenizer_comparison.png')

## Top BPE Merges

In [ ]:
print('First 20 BPE merge rules (most frequent pairs that were merged):')
for i, (pair, merged) in enumerate(list(bpe_tok.merge_rules.items())[:20]):
    print(f'  {i+1:3d}. {repr(pair[0])} + {repr(pair[1])} → {repr(merged)}')

## Token Frequency Distribution

In [ ]:
from collections import Counter
import numpy as np

char_counts = Counter(full_char)
bpe_counts  = Counter(full_bpe)

# Plot Zipf-like frequency distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

char_freqs = sorted(char_counts.values(), reverse=True)
bpe_freqs  = sorted(bpe_counts.values(), reverse=True)

axes[0].loglog(range(1, len(char_freqs)+1), char_freqs, color='#4e79a7')
axes[0].set_title('Char Tokenizer: Token Frequency Distribution (log-log)')
axes[0].set_xlabel('Token rank')
axes[0].set_ylabel('Frequency')

axes[1].loglog(range(1, len(bpe_freqs)+1), bpe_freqs, color='#e15759')
axes[1].set_title('BPE Tokenizer: Token Frequency Distribution (log-log)')
axes[1].set_xlabel('Token rank')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## Conclusion

| | Char | Word | BPE |
|---|---|---|---|
| Vocab size | ~65 | 5,000 | 3,000 |
| Compression | 1.0x | ~5x | ~2.5x |
| OOV problem | None | Severe | Minor |
| Used in GPT-2/3 | No | No | **Yes** |

**Key insight:** BPE strikes the best balance — significantly shorter sequences than character-level (reducing computational cost) while nearly eliminating the OOV problem that plagues word-level tokenizers.